# CRISP Spectropolarimetry — Implementation / CRISP 분광편광 구현

**Paper**: Scharmer, G. B. et al., "CRISP spectropolarimetric imaging of penumbral fine structure", ApJL 689, L69 (2008).

이 노트북은 논문의 관측 체인을 교육적 규모로 재현한다 / This notebook reproduces the observation chain at a pedagogical scale:

1. **Fabry–Pérot transmission** — 단일·이중 에탈론의 투과 함수.
2. **Zeeman splitting + weak-field Stokes profiles** — $I, Q, U, V$ 생성.
3. **Synthetic penumbra filament** — 수평 flux tube + 수직 배경장 모델로 공간 맵 생성.
4. **Center-of-gravity magnetogram** — Stokes $V$로부터 $B_{\rm LOS}$ 복원.
5. **Doppler map** — line-core shift로부터 $v_{\rm LOS}$ 복원; filament tail의 downflow 시연.
6. **Line ratio 6301/6302** — Stenflo의 진단.

Environment: conda env `study-with-ai`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.dpi": 110, "image.cmap": "gray"})

# Physical constants
C_ZEEMAN = 4.67e-13  # ZeemanConst in Å/G when lambda in Å
C_LIGHT = 2.998e5    # km/s

## 1. Fabry–Pérot transmission / 에탈론 투과 함수

$$T(\lambda) = \frac{1}{1 + F \sin^2(\delta/2)}, \quad \delta = \frac{4\pi n t}{\lambda}$$

CRISP은 이중 FP: 낮은-finesse prefilter × 높은-finesse 에탈론.

In [ ]:
def fp_transmission(lam: np.ndarray, nt_um: float, R: float) -> np.ndarray:
    """Fabry–Pérot transmission.

    Args:
        lam: Wavelength array (Å).
        nt_um: Optical thickness n*t (micrometers).
        R: Reflectance of each surface.

    Returns:
        Transmission array.
    """
    F = 4 * R / (1 - R) ** 2
    # convert nt to Å: 1 um = 1e4 Å
    delta = 4 * np.pi * (nt_um * 1e4) / lam
    return 1.0 / (1 + F * np.sin(delta / 2) ** 2)


lam = np.linspace(6300, 6305, 20000)

# High-finesse etalon: narrow passband ~6 pm FWHM
T_hi = fp_transmission(lam, nt_um=700.0, R=0.93)
# Low-finesse prefilter: wider
T_lo = fp_transmission(lam, nt_um=70.0, R=0.50)
T_tot = T_hi * T_lo

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lam, T_hi, label="High-finesse etalon", alpha=0.7)
ax.plot(lam, T_lo, label="Low-finesse prefilter", alpha=0.7)
ax.plot(lam, T_tot, "k-", label="Combined (dual FP)", lw=1.5)
ax.set_xlabel("Wavelength (Å)")
ax.set_ylabel("Transmission")
ax.set_title("CRISP-style dual Fabry–Pérot")
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(6301, 6303)
plt.show()

## 2. Zeeman splitting + weak-field Stokes profiles / Zeeman 분리와 약자장 Stokes 프로파일

Fe I 6301.5 Å ($g=1.67$), Fe I 6302.5 Å ($g=2.5$).

$$V = -C g \lambda^2 B\cos\theta_B \frac{\partial I_0}{\partial \lambda}, \quad Q, U \propto (g \lambda^2 B \sin\theta_B)^2 \frac{\partial^2 I_0}{\partial \lambda^2}$$

In [ ]:
def gaussian_line(lam: np.ndarray, lam0: float, depth: float, width: float) -> np.ndarray:
    """Simple Gaussian absorption line profile."""
    return 1.0 - depth * np.exp(-((lam - lam0) ** 2) / (2 * width ** 2))


def weak_field_stokes(lam: np.ndarray, lam0: float, depth: float, width: float,
                     g_eff: float, B: float, theta_B: float, phi_B: float,
                     v_los: float) -> dict:
    """Weak-field Stokes profiles for a single Zeeman-sensitive line.

    Args:
        lam: Wavelength array (Å).
        lam0: Rest wavelength (Å).
        depth: Line depth.
        width: Doppler width (Å).
        g_eff: Effective Landé factor.
        B: Field strength (Gauss).
        theta_B: Inclination to LOS (rad).
        phi_B: Azimuth in POS (rad).
        v_los: LOS velocity (km/s, positive = redshift).

    Returns:
        Dict of I, Q, U, V arrays.
    """
    # apply Doppler shift
    lam_shift = lam0 * (1 + v_los / C_LIGHT)
    I = gaussian_line(lam, lam_shift, depth, width)
    dI = np.gradient(I, lam)
    d2I = np.gradient(dI, lam)

    dlambda_B = C_ZEEMAN * g_eff * lam0 ** 2 * B  # Å
    V = -dlambda_B * np.cos(theta_B) * dI
    q_amp = 0.25 * (dlambda_B * np.sin(theta_B)) ** 2 * d2I
    Q = q_amp * np.cos(2 * phi_B)
    U = q_amp * np.sin(2 * phi_B)
    return {"I": I, "Q": Q, "U": U, "V": V}

In [ ]:
# Compare two pixel conditions: vertical-field vs horizontal-field
lam_loc = np.linspace(6302.0, 6303.0, 500)
lam0 = 6302.5

vert = weak_field_stokes(lam_loc, lam0, depth=0.55, width=0.04,
                         g_eff=2.5, B=1500, theta_B=np.deg2rad(20), phi_B=0.0, v_los=0.0)
horiz = weak_field_stokes(lam_loc, lam0, depth=0.55, width=0.04,
                          g_eff=2.5, B=1500, theta_B=np.deg2rad(85), phi_B=np.deg2rad(45), v_los=0.0)

fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for name, key in zip(["I", "Q", "U", "V"], ["I", "Q", "U", "V"]):
    ax_ = ax["IQUV".index(name)]
    ax_.plot(lam_loc, vert[key], label="vertical field (θ=20°)")
    ax_.plot(lam_loc, horiz[key], label="horizontal field (θ=85°)")
    ax_.set_title(name)
    ax_.set_xlabel("λ (Å)"); ax_.grid(alpha=0.3)
    if name == "V":
        ax_.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Vertical: large |V|, small Q/U. Horizontal: |V|≈0, Q/U prominent — dark-core signature.")

## 3. Synthetic penumbra filament / 합성 반암부 필라멘트

**Uncombed model**:
- 필라멘트 중심 (dark core): 수평 flux tube + Evershed outflow
- 필라멘트 측면 (bright sides): 수직 배경장, 정지
- 필라멘트 tail: 하향류 + 자기장 약화

Generate a 2D (x, y) map of (intensity, B, θ, v_los) for one filament.

In [ ]:
def filament_map(Nx: int = 160, Ny: int = 40) -> dict:
    """Generate a toy penumbral filament (uncombed model).

    Args:
        Nx: Along-filament pixels (radial direction in penumbra).
        Ny: Across-filament pixels.

    Returns:
        Dict with 2D maps of continuum, B (G), theta (rad), v_los (km/s).
    """
    x = np.arange(Nx) - Nx // 2
    y = np.arange(Ny) - Ny // 2
    X, Y = np.meshgrid(x, y, indexing="xy")

    # Intensity: bright lateral sides, dark core at |y|~0, dark tail near +x end
    core = np.exp(-(Y / 3) ** 2)
    sides = 1 - 0.6 * core
    tail_fade = 1 - 0.5 * np.exp(-((X - Nx // 2 + 10) / 6) ** 2)
    cont = sides * tail_fade
    cont = 0.7 + 0.3 * (cont - cont.min()) / (cont.max() - cont.min())

    # Inclination: nearly horizontal (85°) at core, more vertical (30°) at sides
    theta = np.deg2rad(30 + 55 * core)
    # Field strength ~1.5 kG, dropping at tail
    B = 1500.0 * (1 - 0.3 * np.exp(-((X - Nx // 2 + 10) / 6) ** 2))

    # Flow: Evershed outflow along x+ in the core, downflow in the tail patch
    outflow_x = -3.0 * core                # core channels outflow (toward +x, neg v_los if outflow away from observer)
    # For disk-center view near an outer penumbra edge, outflow contributes LOS component
    v_outflow = outflow_x
    # Downflow in tail region
    tail_region = np.exp(-((X - Nx // 2 + 10) / 5) ** 2 - (Y / 3) ** 2)
    v_downflow = 5.0 * tail_region         # positive LOS = redshift
    v_los = v_outflow + v_downflow
    return {"cont": cont, "B": B, "theta": theta, "v_los": v_los}


fmap = filament_map(160, 40)

fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
ax[0].imshow(fmap["cont"]); ax[0].set_ylabel("continuum")
im1 = ax[1].imshow(np.rad2deg(fmap["theta"]), cmap="viridis")
plt.colorbar(im1, ax=ax[1]).set_label("deg")
ax[1].set_ylabel("θ_B (inclination)")
im2 = ax[2].imshow(fmap["B"], cmap="viridis")
plt.colorbar(im2, ax=ax[2]).set_label("G")
ax[2].set_ylabel("|B|")
im3 = ax[3].imshow(fmap["v_los"], cmap="RdBu_r", vmin=-5, vmax=5)
plt.colorbar(im3, ax=ax[3]).set_label("km/s")
ax[3].set_ylabel("v_LOS"); ax[3].set_xlabel("x (radial)")
plt.tight_layout(); plt.show()
print("Red patch near x ~ 70 = filament-tail downflow; blueshifted core = Evershed outflow.")

## 4. Synthesize Stokes cube & recover magnetogram / Stokes 큐브 합성과 magnetogram 복원

각 픽셀에 weak-field Stokes를 적용해 $(x, y, \lambda)$ Stokes cube를 만든 뒤, **center-of-gravity**로 $B_{\rm LOS}$를 역산.

In [ ]:
def stokes_cube(fmap: dict, lam_grid: np.ndarray, lam0: float = 6302.5,
                g_eff: float = 2.5) -> dict:
    """Build (I, Q, U, V) cube over 2D filament map."""
    Ny, Nx = fmap["cont"].shape
    shape = (Ny, Nx, len(lam_grid))
    I = np.zeros(shape); Q = np.zeros(shape); U = np.zeros(shape); V = np.zeros(shape)
    for j in range(Ny):
        for i in range(Nx):
            s = weak_field_stokes(
                lam_grid, lam0, depth=0.55, width=0.04, g_eff=g_eff,
                B=fmap["B"][j, i], theta_B=fmap["theta"][j, i], phi_B=0.0,
                v_los=fmap["v_los"][j, i],
            )
            I[j, i] = s["I"] * fmap["cont"][j, i]
            Q[j, i] = s["Q"]; U[j, i] = s["U"]; V[j, i] = s["V"]
    return {"I": I, "Q": Q, "U": U, "V": V, "lam": lam_grid}


lam_grid = np.linspace(6302.0, 6303.0, 41)  # 25 mÅ sampling
cube = stokes_cube(fmap, lam_grid)
print("Stokes cube shape:", cube["I"].shape)

In [ ]:
def cog_magnetogram(cube: dict, lam0: float = 6302.5, g_eff: float = 2.5) -> np.ndarray:
    """Center-of-gravity magnetogram from Stokes V.

    Uses the weak-field proportionality:
        B_LOS ∝ ∫ V(λ)(λ-λ0) dλ  /  [ (C * g * λ0²) * ∫ (1-I/I_c) dλ ]
    """
    lam = cube["lam"]
    I = cube["I"]; V = cube["V"]
    Ic = I.max(axis=-1, keepdims=True)
    absorb = (Ic - I)
    num = np.trapz(V * (lam - lam0), lam, axis=-1)
    den = np.trapz(absorb, lam, axis=-1)
    return num / (C_ZEEMAN * g_eff * lam0 ** 2 * den + 1e-20)


B_los_truth = fmap["B"] * np.cos(fmap["theta"])
B_los_recon = cog_magnetogram(cube)

fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
im0 = ax[0].imshow(B_los_truth, cmap="RdBu_r", vmin=-1600, vmax=1600)
ax[0].set_title("True B_LOS = |B| cos θ")
plt.colorbar(im0, ax=ax[0], label="G")
im1 = ax[1].imshow(B_los_recon, cmap="RdBu_r", vmin=-1600, vmax=1600)
ax[1].set_title("Recovered B_LOS from CoG of Stokes V")
plt.colorbar(im1, ax=ax[1], label="G")
plt.tight_layout(); plt.show()
print("Dark core (|y|~0) shows near-zero B_LOS in both maps — horizontal field signature.")

## 5. Doppler map from line-core shift / 선 중심 변위로부터 도플러 맵

각 픽셀에서 Stokes I의 최소값 위치(line core)를 포물선 피팅으로 찾아 $v_{\rm LOS}$를 복원.

In [ ]:
def line_core_doppler(cube: dict, lam0: float = 6302.5) -> np.ndarray:
    """Line-core Doppler map via parabolic fit."""
    lam = cube["lam"]
    I = cube["I"]
    idx = np.argmin(I, axis=-1)
    v = np.zeros(idx.shape)
    for j in range(I.shape[0]):
        for i in range(I.shape[1]):
            k = idx[j, i]
            if 0 < k < len(lam) - 1:
                y0, y1, y2 = I[j, i, k - 1], I[j, i, k], I[j, i, k + 1]
                denom = y0 - 2 * y1 + y2
                delta = 0.5 * (y0 - y2) / (denom if denom != 0 else 1e-20)
                lam_core = lam[k] + delta * (lam[1] - lam[0])
            else:
                lam_core = lam[k]
            v[j, i] = (lam_core - lam0) / lam0 * C_LIGHT
    return v


v_recon = line_core_doppler(cube)

fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
im0 = ax[0].imshow(fmap["v_los"], cmap="RdBu_r", vmin=-5, vmax=5)
ax[0].set_title("True v_LOS"); plt.colorbar(im0, ax=ax[0], label="km/s")
im1 = ax[1].imshow(v_recon, cmap="RdBu_r", vmin=-5, vmax=5)
ax[1].set_title("Recovered v_LOS (line-core fit)"); plt.colorbar(im1, ax=ax[1], label="km/s")
plt.tight_layout(); plt.show()
print("The tail redshift patch — the paper's downflow signature — is cleanly recovered.")

## 6. Line ratio 6301/6302 — Stenflo diagnostic / 선 비율 진단

약자장 극한에서 $V_{6302}/V_{6301} = g_{6302}/g_{6301} = 1.50$.
강한 자기장에서는 saturation으로 이 비율이 감소한다.

In [ ]:
def v_amplitude(lam0: float, g_eff: float, B: float, theta_B: float) -> float:
    """Peak |V| amplitude in weak-field limit for a Gaussian line."""
    lam = np.linspace(lam0 - 0.5, lam0 + 0.5, 2000)
    stokes = weak_field_stokes(lam, lam0, depth=0.55, width=0.04,
                               g_eff=g_eff, B=B, theta_B=theta_B, phi_B=0.0, v_los=0.0)
    return np.max(np.abs(stokes["V"]))


Bs = np.linspace(100, 3000, 15)
ratios = []
for B in Bs:
    v1 = v_amplitude(6301.5, 1.67, B, np.deg2rad(30))
    v2 = v_amplitude(6302.5, 2.5, B, np.deg2rad(30))
    ratios.append(v2 / v1)
ratios = np.array(ratios)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(Bs, ratios, "o-")
ax.axhline(1.5, color="r", ls="--", label="Weak-field limit (g ratio)")
ax.set_xlabel("B (G)"); ax.set_ylabel("V(6302) / V(6301) amplitude")
ax.set_title("Stenflo line-ratio diagnostic")
ax.legend(); ax.grid(alpha=0.3)
plt.show()
print("In pure weak-field regime, ratio is flat at 1.50. Our Gaussian profile is linear here; "
      "a Voigt + saturation model would bend downward beyond B ~ 1500 G.")

## 7. Summary / 요약

- **이중 Fabry–Pérot**가 6 pm 수준 bandpass와 side-order 제거를 동시 달성.
- **Weak-field Stokes $V, Q, U$** 가 필라멘트 중심(수평장)과 측면(수직장)에서 논문이 관측한 신호 구분을 재현.
- **Center-of-gravity magnetogram**이 $B_{\rm LOS}$를 2D 맵으로 복원.
- **Line-core Doppler 맵**이 필라멘트 tail의 하향류 패치를 복원 — 논문의 핵심 발견.
- **Stenflo line-ratio** 개념을 수치로 시연.

Takeaways: the dual FP delivers the ~6 pm bandpass needed for CRISP; weak-field Stokes profiles reproduce the horizontal-core vs vertical-sides signature; CoG magnetograms and line-core Doppler maps recover both the uncombed field structure and the tail downflow — the two principal discoveries of the paper.